In [1]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder
from sklearn.svm import SVC

In [2]:
df = pd.read_csv('loan_data.csv')
df

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44995,27.0,male,Associate,47971.0,6,RENT,15000.0,MEDICAL,15.66,0.31,3.0,645,No,1
44996,37.0,female,Associate,65800.0,17,RENT,9000.0,HOMEIMPROVEMENT,14.07,0.14,11.0,621,No,1
44997,33.0,male,Associate,56942.0,7,RENT,2771.0,DEBTCONSOLIDATION,10.02,0.05,10.0,668,No,1
44998,29.0,male,Bachelor,33164.0,4,RENT,12000.0,EDUCATION,13.23,0.36,6.0,604,No,1


In [3]:
X = df.drop(columns = 'loan_status')
y= df.loan_status

In [4]:
xtrain, xtest, ytrain, ytest = train_test_split(X,y,train_size=0.8, random_state=42)

In [5]:
#num_cols = X.select_dtypes(include='number').columns
obj_cols = X.select_dtypes(include='object').columns

C:\Users\DELL\AppData\Local\Temp\ipykernel_5780\1433802163.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = X.select_dtypes(include='object').columns


In [6]:
X[obj_cols].nunique()

person_gender                     2
person_education                  5
person_home_ownership             4
loan_intent                       6
previous_loan_defaults_on_file    2
dtype: int64

In [7]:
# person_gender                     2
# person_education                  5
# person_home_ownership             4
# loan_intent                       6
# previous_loan_defaults_on_file    2
# 2 unique values and less feature values do OneHotEncoding and more unique values do OrdinalEncoding

In [8]:
X['person_education'].unique()

<ArrowStringArray>
['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate']
Length: 5, dtype: str

In [9]:
order = ['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate']

In [10]:
obj_cols.drop(['person_education'])

Index(['person_gender', 'person_home_ownership', 'loan_intent',
       'previous_loan_defaults_on_file'],
      dtype='str')

In [11]:
# preprocessing = ColumnTransformer(
#     transformers=[
#         ('onehot_encoder', OneHotEncoder(handle_unknown='ignore'),obj_cols.drop('person_education')),
#         ('ordinal_encoder', OrdinalEncoder(categories=[order],handle_unknown='use_encoded_value',unknown_value=-1),['person_education']))
#     ],remainder='passthrough'
# )

# main_pipeline = Pipeline(
#     steps=[
#         ('preprocessing', preprocessing),
#         ('model', DecisionTreeClassifier(random_state=42))
#     ]
# )

# grid_search_cv = GridSearchCV(
#     estimator = main_pipeline,
#     param_grid = {
#         'model__criterion':['gini','entropy'], #model__ is used to access the parameters of the model in the pipeline i.e go inside model and access the parameters of DecisionTreeClassifier
#         'model__max_depth':[None,5,10,50,100], #if we use algorithm no need to use model__ but if we use pipeline then we need to use model__ to access the parameters of the model in the pipeline
#         'model__min_samples_split':[2,5,7,10],
#         'model__min_samples_leaf':[1,3,5,7,10],
#         'model__splitter':['best', 'random']
#     },
# verbose=1)
# grid_search_cv.fit(xtrain,ytrain)

In [12]:
# preprocessing = ColumnTransformer(
#     transformers=[
#         ('onehot_encoder', OneHotEncoder(handle_unknown='ignore'), obj_cols.drop('person_education')),
#         ('ordinal_encoder', OrdinalEncoder(categories=[order],handle_unknown='use_encoded_value',unknown_value=-1), ['person_education'])
#     ],
#     remainder='passthrough'
# )

# main_pipeline = Pipeline(
#     steps=[
#         ('preprocessing', preprocessing),
#         ('model', DecisionTreeClassifier(random_state=42))
#     ]
# )

# grid_search_cv = GridSearchCV(
#     estimator = main_pipeline,
#     param_grid = {
#         'model__criterion':['gini','entropy'],
#         'model__max_depth':[None,5,10,50,100],
#         'model__min_samples_split':[2,5,7,10],
#         'model__min_samples_leaf':[1,3,5,7,10],
#         'model__splitter':['best', 'random']
#     },
#     verbose=1,n_jobs=-1
# )
# grid_search_cv.fit(xtrain,ytrain)

In [ ]:
preprocessing = ColumnTransformer(
    transformers=[
        ('onehot_encoder', OneHotEncoder(handle_unknown='ignore'), obj_cols.drop('person_education')),
        ('ordinal_encoder', OrdinalEncoder(categories=[order],handle_unknown='use_encoded_value',unknown_value=-1), ['person_education'])
    ],
    remainder='passthrough'
)

main_pipeline = Pipeline(
    steps=[
        ('preprocessing', preprocessing),
        ('model', SVC(random_state=42))
    ]
)

grid_search_cv = GridSearchCV(
    estimator = main_pipeline,
    param_grid = {
        'model__C':[0.01,0.1,1.0,10,100],
        'model__kernel':['linear', 'poly', 'rbf', 'sigmoid']
    },
    verbose=1,n_jobs=-1
)
grid_search_cv.fit(xtrain,ytrain)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


In [ ]:
grid_search_cv.best_estimator_

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('onehot_encoder', ...), ('ordinal_encoder', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output o

In [ ]:
grid_search_cv.best_params_

{'model__criterion': 'entropy',
 'model__max_depth': 10,
 'model__min_samples_leaf': 1,
 'model__min_samples_split': 2,
 'model__splitter': 'best'}

In [ ]:
grid_search_cv.cv_results_

{'mean_fit_time': array([0.54374146, 0.409128  , 0.61099949, 0.41524334, 0.60886197,
        0.37530193, 0.57531104, 0.42309184, 0.56812048, 0.39872737,
        0.58563905, 0.4413805 , 0.57774992, 0.37070189, 0.61188817,
        0.38660522, 0.58796735, 0.44817767, 0.60301881, 0.34639721,
        0.58523035, 0.41366477, 0.57727513, 0.40404773, 0.58277321,
        0.34120073, 0.56238523, 0.39315248, 0.58403449, 0.36240091,
        0.56451273, 0.34377179, 0.56037526, 0.35694542, 0.55732369,
        0.34402924, 0.53643465, 0.37387118, 0.57559781, 0.42108107,
        0.44291949, 0.39717011, 0.46867933, 0.3680922 , 0.43731089,
        0.40598702, 0.46529455, 0.41918864, 0.44871373, 0.38598576,
        0.45986738, 0.4146534 , 0.4955667 , 0.43688893, 0.43908453,
        0.39047408, 0.46885509, 0.39908624, 0.47677097, 0.39054608,
        0.469807  , 0.4341321 , 0.47611213, 0.40271573, 0.46144843,
        0.37644272, 0.4766161 , 0.38656268, 0.44496512, 0.39716744,
        0.46992064, 0.37804155,

In [ ]:
results = pd.DataFrame(grid_search_cv.cv_results_)
results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__criterion,param_model__max_depth,param_model__min_samples_leaf,param_model__min_samples_split,param_model__splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.543741,0.017841,0.053497,0.008516,gini,None,1,2,best,"{'model__criterion': 'gini', 'model__max_depth...",0.896806,0.900972,0.899444,0.901250,0.895694,0.898833,0.002225,316
1,0.409128,0.037924,0.069182,0.017541,gini,None,1,2,random,"{'model__criterion': 'gini', 'model__max_depth...",0.888333,0.891111,0.884583,0.892639,0.889722,0.889278,0.002749,355
2,0.610999,0.024249,0.053957,0.011334,gini,None,1,5,best,"{'model__criterion': 'gini', 'model__max_depth...",0.895139,0.901806,0.900000,0.901667,0.901250,0.899972,0.002499,301
3,0.415243,0.036105,0.075982,0.011748,gini,None,1,5,random,"{'model__criterion': 'gini', 'model__max_depth...",0.895278,0.897778,0.890000,0.889306,0.895417,0.893556,0.003315,349
4,0.608862,0.011143,0.052521,0.007809,gini,None,1,7,best,"{'model__criterion': 'gini', 'model__max_depth...",0.897917,0.900556,0.900556,0.903472,0.901667,0.900833,0.001807,292
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,0.349646,0.019543,0.072172,0.006490,entropy,100,10,5,random,"{'model__criterion': 'entropy', 'model__max_de...",0.902222,0.909444,0.899861,0.904583,0.909722,0.905167,0.003904,186
396,0.613393,0.056261,0.061059,0.008761,entropy,100,10,7,best,"{'model__criterion': 'entropy', 'model__max_de...",0.908194,0.915139,0.912778,0.912917,0.912639,0.912333,0.002264,81
397,0.387402,0.017768,0.077166,0.018856,entropy,100,10,7,random,"{'model__criterion': 'entropy', 'model__max_de...",0.902222,0.909444,0.899861,0.904583,0.909722,0.905167,0.003904,186
398,0.622819,0.064556,0.052942,0.008182,entropy,100,10,10,best,"{'model__criterion': 'entropy', 'model__max_de...",0.908194,0.915139,0.912778,0.912917,0.912639,0.912333,0.002264,81


In [ ]:
results.sort_values(by='rank_test_score')

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__criterion,param_model__max_depth,param_model__min_samples_leaf,param_model__min_samples_split,param_model__splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
302,0.558305,0.025572,0.069652,0.011940,entropy,10,5,10,best,"{'model__criterion': 'entropy', 'model__max_de...",0.918472,0.927361,0.914444,0.921250,0.920417,0.920389,0.004204,1
300,0.563666,0.035018,0.061651,0.007456,entropy,10,5,7,best,"{'model__criterion': 'entropy', 'model__max_de...",0.918472,0.927361,0.914444,0.921250,0.920417,0.920389,0.004204,1
298,0.557579,0.022663,0.067241,0.014887,entropy,10,5,5,best,"{'model__criterion': 'entropy', 'model__max_de...",0.918472,0.927361,0.914444,0.921250,0.920417,0.920389,0.004204,1
280,0.566087,0.075864,0.065647,0.015842,entropy,10,1,2,best,"{'model__criterion': 'entropy', 'model__max_de...",0.919583,0.926111,0.914861,0.920833,0.920556,0.920389,0.003583,1
296,0.561783,0.047348,0.062086,0.009667,entropy,10,5,2,best,"{'model__criterion': 'entropy', 'model__max_de...",0.918472,0.927361,0.914444,0.921250,0.920417,0.920389,0.004204,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,0.378042,0.002573,0.078127,0.004641,gini,5,7,10,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872222,0.871667,0.872500,0.872639,0.872417,0.000461,393
77,0.370802,0.027165,0.064802,0.005994,gini,5,10,7,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872222,0.871667,0.872500,0.872222,0.872333,0.000451,397
79,0.407490,0.060033,0.070724,0.011259,gini,5,10,10,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872222,0.871667,0.872500,0.872222,0.872333,0.000451,397
73,0.389266,0.020138,0.071000,0.006934,gini,5,10,2,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872222,0.871667,0.872500,0.872222,0.872333,0.000451,397


In [ ]:
#do the same with logistic regression and random forest 